# Ch.4  時系列データの異常を検知する

【講師用完全版】

| 項目 | 内容 |
|------|------|
| 使うデータ | 人工的な時系列データ（工場センサーのイメージ） |
| 演習時間 | 35分（問3まで完了で合格） |
| ゴール | LinearRegression を使って「過去の傾向からのズレ」を数値化し、異常点を検出する |

---
### 📌 講師メモ：時間配分と時間が押した場合の対応

| 区分 | 内容 | 目安時間 |
|------|------|---------|
| 座学（concept_ch4） | 時系列・ラグ特徴量・LinearRegression・しきい値 | 25分 |
| 演習（STEP1〜STEP2） | 可視化・ラグ特徴量作成 | 13分 |
| 演習（問1〜問3） | モデル学習・残差計算・しきい値判定・グラフ | 22分 |
| 問4（発展） | パーセンタイル比較 | 余り次第 |

**時間が押した場合：**
- STEP2（ラグ特徴量）はコードを講師がデモして受講生はコピーでも可
- 問3-2（グラフのハイライト）まで完了をゴールとする
- 問4（発展）はスキップ可
- **最優先は「赤い点が time 80〜90 に集まるグラフを表示すること」**

---
## Copilot の使い方

URL: https://copilot.microsoft.com

Copilot への伝え方のコツ（5点セット）：
```
【知りたいこと】〇〇をしたい / 〇〇を計算したい
【使うライブラリ】pandas / matplotlib / scikit-learn
【データの形】df という DataFrame、120行×3列（time / value / lag_1）
【環境】Python 3.8.6、JupyterLab
【困っていること】どう書けばよいかわからない
```

💡 Ch.4 の一言アドバイス：
「shift() や predict() など、聞いたことのないメソッドは
 『df の value 列を1行ずらして lag_1 列に入れたい、pandas でどう書くか』のように
 列名・操作・結果をセットで伝えましょう。」

時系列の操作は慣れないと難しいですが、**やりたいことを日本語で具体的に説明する**だけで
Copilot が書いてくれます。

---
## 準備  ライブラリとデータを生成する

⛔ 注意 このセルを実行するとデータが生成されます。必ず最初に実行してください。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
from sklearn.linear_model import LinearRegression

# 再現性のためシードを固定して、工場センサーのような時系列データを生成する
np.random.seed(42)
n       = 120
time    = np.arange(n)
base    = 50 + 0.3 * time + np.random.normal(0, 2, n)  # 正常な緩やかな上昇トレンド
# 異常期間（80〜90 番目）にスパイクを追加
base[80:90] += np.random.uniform(20, 35, 10)
df = pd.DataFrame({"time": time, "value": base})
print(f"データ件数: {len(df)} 行")
df.head()

---
## なぜ「予測との差」で異常を検知するのか

異常検知の考え方（LinearRegression を使う方法）：

1. 正常期間のデータでモデルを学習させる（「正常とはこういうもの」を学ぶ）
2. 全期間でモデルが「予測する値」を計算する
3. 「実際の値」と「予測値」のズレ（残差）を計算する
4. ズレが大きい点 = 正常パターンから外れた = 「異常の疑い」

実務での活用：
- 工場機器の温度・振動センサーで「故障の前兆」を検知
- 売上データで「急激な増減（キャンペーン効果 or 問題）」を検知

---
## STEP 1  データを「目で」確認する

どこにスパイク（急激な変動）があるかをグラフで見て、感覚を掴みましょう。
機械に任せる前に、「人間の目でも明らかな異常があるか」を確認することが大切です。

---

Q1-1：折れ線グラフで時系列データを表示してください

time を X 軸、value を Y 軸にして、データ全体の推移を確認してください。

💡 ヒント Copilot への相談例：
「pandas の DataFrame の 2 列を使って折れ線グラフを描きたい。X軸とY軸のラベルも付けたい」

In [ ]:
# time を X 軸、value を Y 軸にした折れ線グラフを描いて、全体の推移を確認する
plt.figure(figsize=(12, 4))
plt.plot(df["time"], df["value"])
plt.title("センサー値の時系列推移")
plt.xlabel("time")
plt.ylabel("value")
plt.show()

---
### 📊 出力結果の解説

| 確認するポイント | 見るべきこと |
|----------------|------------|
| 全体トレンド | 緩やかに右肩上がり（正常なトレンド） |
| スパイク | time 80〜90 あたりに大きな突出 |
| 正常期間 | 0〜79 はトレンドに沿って安定 |

📌 ポイント 目視でもスパイクが確認できる。
しかし「どこからが異常か」の境界は曖昧 → 機械学習で「数値で定量化」する。
自動化する前に「人間の目で確認できるか」を必ず行う習慣を付ける。

📌 講師メモ: 「80〜90 番目だとわかりますか？」と受講生に確認させる。

---
### STEP 1 の気づき（AI 禁止：1 行でOK）

Q：スパイク（急激な変動）は何番目付近に見えましたか？また、正常期間はどのような形をしていましたか？

>

✅ （模範解答）time 80〜90 付近に大きなスパイクが見える。正常期間（0〜79）は緩やかな右肩上がりのトレンドで安定している。

📌 講師メモ：「何番目に異常がありそうですか？」と受講生に声がけ。後の機械学習の結果と照合させる布石にする。

---
## STEP 2  「過去の値」を新しい列として追加する（ラグ特徴量）

なぜラグ特徴量が必要か：
LinearRegression に「時刻 t の値」を予測させるには、「時刻 t-1 の値」を入力として使います。
「前の時点の値」を新しい列として追加することを「ラグ特徴量を作る」と言います。

---

Q2-1：1 時点前の値（lag_1）を追加してください

`shift()` を使って、1 行前の value 列の値を新しい列として追加してください。

💡 ヒント Copilot への相談例：
「pandas で特定の列の値を 1 行分だけずらした新しい列を作りたい。欠損が出るので dropna で削除する」

---
### shift() を使うと何が起きるか

`shift(1)` を実行すると、先頭の行が **NaN（空欄）** になります。

```
実行前:          実行後:
time  value     time  value  lag_1
  0   50.6        0   50.6   NaN   ← 先頭行は「1つ前」がないのでNaN
  1   51.3        1   51.3   50.6
  2   49.8        2   49.8   51.3
```

NaN のまま計算できないので `dropna()` で先頭行を削除します。
**1行減るのは正常な動作です。**

In [ ]:
# 1 時点前の値をラグ特徴量（lag_1）として新しい列に追加して、欠損行を削除する
df["lag_1"] = df["value"].shift(1)
df = df.dropna().reset_index(drop=True)   # 1行目が NaN になるため削除
print(f"ラグ追加後: {len(df)} 行")
df.head()

---
### 📊 出力結果の解説

```
   time      value      lag_1
0     1  50.976638  50.606186   <- time=1 の lag_1 は time=0 の value
1     2  55.081445  50.976638
...
```

| 列 | 意味 |
|----|------|
| value | 現時点（t）のセンサー値 |
| lag_1 | 1 時点前（t-1）のセンサー値 |

📌 ポイント lag_1 は「直前の値」を特徴量として使うことで、モデルが「過去の傾向」を学べる。
時系列データに機械学習を使うときの基本テクニック。

📌 講師メモ: 「なぜ 1 行目が削除されるのか？」を問い返す（shift() で 1 行目が NaN になるから）。

---
### STEP 2 の気づき（AI 禁止：1 行でOK）

Q：`df.head()` を確認して、lag_1 列に何が入っているか説明してください。先頭行が NaN だった場合はどうしましたか？

>

✅ （模範解答）lag_1 列には 1 行前の value 値が入っている（例：time=1 の lag_1 は time=0 の value）。先頭行は 1 つ前のデータがないため NaN になり、dropna() で削除した。

📌 講師メモ：「なぜ先頭行が NaN になるのか？」を問い返す。shift(1) の動作を図で補足すると理解が深まる。

---
## 問1  正常期間だけでモデルを学習させる ★☆☆

実務でのイメージ：
「故障前の正常データでモデルを作り、そのモデルで予測した値と現在値のズレを異常スコアとして使う」

---

Q：最初の 80 点（正常期間）だけで LinearRegression のモデルを学習させてください

- 正常期間：time が 0〜79（インデックスで最初の 80 行）
- X（入力）：lag_1 列
- y（答え）：value 列

💡 ヒント Copilot への相談例：
「pandas DataFrame の特定の行だけを取り出して、LinearRegression のモデルを学習させたい」

In [ ]:
# 最初の 80 行（正常期間）だけを取り出して、LinearRegression を学習させる
train   = df[df["time"] < 80]                     # 正常期間のデータ（80件）
X_train = train[["lag_1"]]                        # 入力：1 時点前の値
y_train = train["value"]                          # 答え：現在の値

model = LinearRegression()
model.fit(X_train, y_train)
print("学習完了")

---
### 📊 出力結果の解説

```
学習完了
```

📌 ポイント 正常期間（time 0〜79）で学習することで、モデルは「正常とはどういう動きか」を学ぶ。
異常期間（80〜90）のデータは学習に使わない = テスト用に残しておく。

✅ （模範解答）`train = df[df["time"] < 80]` で正常期間を取り出し、`model.fit(X_train, y_train)` で学習。
エラーなく「学習完了」と表示されれば成功。

📌 講師メモ: 「なぜ正常期間だけで学習するのか？」を問い返す（正常の基準を学ばせるため）。

---
### 問1 の気づき（AI 禁止：1 行でOK）

Q：正常期間（time 0〜79）だけでモデルを学習させました。なぜ異常期間のデータを学習に使わないのでしょうか？

>

✅ （模範解答）異常期間のデータを学習に含めると、モデルが「異常も正常」と学んでしまい、残差（異常スコア）が小さくなって検知できなくなるから。「正常の基準」を学ばせるために正常データだけを使う。

📌 講師メモ：「もし異常データも含めて学習したらどうなるでしょう？」と問い返す。異常検知の根本的な考え方を定着させる。

---
## 問2  全期間で予測して「ズレ（残差）」を計算する ★★☆

実務でのイメージ：
「正常時の動きを学んだモデルで全データの値を予測し、実際の値との差が大きい点 = 異常候補とする」

---

Q2-1：全期間のデータでモデルが予測した値と、実際の値との差（残差）を計算してください

1. `model.predict(df[["lag_1"]])` で全 119 点の予測値を計算する
2. `df["value"] - 予測値` で残差（ズレ）を計算して `df["residual"]` に入れる
3. 残差の絶対値 `abs(残差)` を `df["score"]` に入れる

💡 ヒント Copilot への相談例：
「scikit-learn の線形回帰モデルで全期間の予測値を計算して、実際の値との差（残差）と絶対値を新しい列に追加したい」

In [ ]:
# 全期間の予測値を計算して、実際の値との差（残差）と異常スコア（絶対値）を追加する
df["pred"]     = model.predict(df[["lag_1"]])      # 全期間の予測値
df["residual"] = df["value"] - df["pred"]          # 残差（実際 - 予測）
df["score"]    = df["residual"].abs()              # 異常スコア = 残差の絶対値
print(df[["time","value","pred","score"]].tail(20))

---
### 📊 出力結果の解説

```
     time      value       pred      score
80     80  78.23 ...   55.6 ...   22.6 ...   <- スパイク開始点で score が急増
...
90     90  51.4 ...    55.0 ...    3.6 ...   <- 正常に戻ると score が小さくなる
```

| 確認するポイント | 見るべきこと |
|----------------|------------|
| score が大きい行 | time 80〜90 付近 → スパイク期間 |
| score が小さい行 | 正常期間（0〜79）は 0〜5 程度 |

📌 ポイント score が「実際の値」と「正常時の予測値」のズレを数値化している。
この score が大きい点が「異常の疑いがある点」。

📌 講師メモ: 「score が 0 に近いほど正常、大きいほど異常。どこから異常と判断するか？」と問い返す。

---
## 問2-2  異常スコアの推移を確認する ★★☆

Q：score 列（異常スコア）の折れ線グラフを表示して、どの time 帯でスコアが急増しているか確認してください

time を X 軸、score を Y 軸にして、スパイクが見えるか確認します。
STEP 1 で目視したグラフと見比べてみましょう。

💡 ヒント Copilot への相談例：
「pandas の DataFrame の 2 列を使って折れ線グラフを描きたい。X軸とY軸のラベルも付けたい」

In [ ]:
# 異常スコアの時系列推移を折れ線グラフで確認する
plt.figure(figsize=(12, 4))
plt.plot(df["time"], df["score"])
plt.title("異常スコア（残差の絶対値）の推移")
plt.xlabel("time")
plt.ylabel("score（異常スコア）")
plt.show()

---
### 📊 出力結果の解説

| 確認するポイント | 見るべきこと |
|----------------|------------|
| スコアが急増する time 帯 | 80〜90 付近でスコアが大きくジャンプ |
| 正常期間のスコア | 0〜79 は 0〜5 程度で安定 |

📌 ポイント このグラフがあることで「しきい値をどこに設定すればよいか」が視覚的にわかる。正常期間のスコアがほぼフラットで、80〜90 だけが突出している = モデルが正常と異常を区別できている証拠。

📌 講師メモ：「このグラフを見て、しきい値を何点くらいに設定したいですか？」と受講生に問い返す。

---
### 問2-2 の気づき（AI 禁止：1 行でOK）

Q：スコアが急増している time 帯はどこでしたか？STEP 1 の折れ線グラフで目視したスパイクの位置と一致していましたか？

>

✅ （模範解答）time 80〜90 付近でスコアが大幅に急増していた。STEP 1 の折れ線グラフでスパイクが見えた場所と一致している。正常期間（0〜79）のスコアは 0〜5 程度と低い。

📌 講師メモ：目視の結果とスコアグラフが一致しているか確認させる。「EDA → 機械学習の流れが繋がっている」ことを意識させる。

---
## 問3  しきい値で異常を判定してグラフに表示する ★★☆（ゴール）

実務でのイメージ：
「異常スコアが一定値以上の点だけをアラートとして抽出し、グラフで確認する」

---

Q3-1：正常期間（最初の 80 点）の score の 95パーセンタイルをしきい値にして、
そのしきい値を超えた点を「異常」と判定してください

1. `np.percentile(正常期間のscore, 95)` でしきい値を計算する
2. `df["score"] > しきい値` で True/False の列を作る（`df["is_anomaly"]`）

💡 ヒント Copilot への相談例：
「numpy で特定の列の 95 パーセンタイルを計算したい。pandas で条件に合う行に True を付けたい」

In [ ]:
# 正常期間の 95 パーセンタイルをしきい値として、それを超えた点を異常とフラグする
normal_score = df[df["time"] < 80]["score"]
threshold    = np.percentile(normal_score, 95)       # 正常期間の上位 5% を基準にする
df["is_anomaly"] = df["score"] > threshold
print(f"しきい値: {threshold:.2f}")
print(f"異常判定件数: {df['is_anomaly'].sum()} 件")

---
### 📊 出力結果の解説

```
しきい値: 4.32
異常判定件数: 9 件
```

📌 ポイント 「正常期間の上位 5%」をしきい値にすることで、正常時のノイズを除外できる。
パーセンタイルを 90 にすれば感度が上がり（異常をより広く検出）、
99 にすれば感度が下がる（明らかな異常だけを検出）。

✅ （模範解答）しきい値は正常期間の score の 95 パーセンタイル（目安 4〜6 程度）。
異常判定件数は 9〜10 件程度（スパイク期間 80〜90 の 10 点に相当）。

📌 講師メモ: 「しきい値を変えると何が変わるか？」→ 問4（発展）に接続。

---
### 問3-1 の気づき（AI 禁止：1 行でOK）

Q：異常判定件数は何件でしたか？スパイク期間（time 80〜90）の 10 点のうち何件が検出できたと思いますか？

>

✅ （模範解答）異常判定件数は 9 件（95パーセンタイルしきい値の場合）。スパイク期間 10 点のうち 9 点を検出できた。

📌 講師メモ：「9 件を聞いて、スパイク期間（80〜90）の 10 点とほぼ一致しているね、と気づかせる。1 件だけ見逃すのはなぜか？」と問い返す。

---
Q3-2：グラフに正常点と異常点を色分けして表示してください

- 正常点：通常の折れ線グラフ
- 異常点（is_anomaly=True の点）：赤い点（散布図）で重ねて表示

💡 ヒント Copilot への相談例：
「matplotlib で折れ線グラフを描いて、条件に合う点だけを赤い点でハイライトしたい」

In [ ]:
# 全体を折れ線で描き、異常判定された点だけを赤い点でハイライト表示する
anomalies = df[df["is_anomaly"]]
plt.figure(figsize=(12, 4))
plt.plot(df["time"], df["value"], label="正常")
plt.scatter(anomalies["time"], anomalies["value"], color="red", zorder=5, label="異常")
plt.axhline(threshold + df["pred"].mean(), color="orange", linestyle="--", label="しきい値")
plt.title("時系列の異常検知結果")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

---
### 📊 出力結果の解説

| グラフ要素 | 意味 |
|----------|------|
| 青い折れ線 | センサー値の推移 |
| 赤い点 | 異常と判定された点（time 80〜90 付近に集中） |
| オレンジ破線 | しきい値のイメージ |

📌 ポイント 目視でスパイクがあった time 80〜90 に、赤い点が集まっていれば成功。
「コードで描いたグラフ」と「人間の目視」の結果が一致 = モデルが正常に動いている。

📌 講師メモ: 「赤い点はだいたいどこに出ましたか？最初に目で見たスパイクと合っていますか？」と問い返す。

✅ （模範解答）time 80〜90 付近にほぼすべての赤い点が集中する。
目視での異常（スパイク期間）と機械の判定が一致している = 異常検知モデルが機能している。

---
### 問3-2 の気づき（AI 禁止：1 行でOK）

Q：赤い点はどこに集まっていましたか？STEP 1 で目視したスパイクの位置と一致していましたか？

>

✅ （模範解答）赤い点が time 80〜90 付近に集中していた。STEP 1 の目視・問2-2のスコアグラフと完全に一致。人間の直感と機械の判断が一致した。

📌 講師メモ：「STEP 1 → 問2-2 → 問3-2 の全体の流れが一致していることを強調する。EDA（目視）→ 異常スコア化（数値）→ 自動検出（グラフ）という流れが完結。」

---
## STEP 3  振り返り （AI 禁止）

⛔ 注意 AI は使わないこと。自分の言葉で書いてください。

Q：LinearRegression を使った異常検知の仕組みを、自分の言葉で説明してください

>

Q：このモデルを実務（工場・在庫管理・売上監視など）でどう使えそうですか？

>

---
### 📌 講師メモ（振り返りの模範解答）

（模範解答 1）異常検知の仕組み：
「正常期間のデータで線形回帰を学習し、モデルが予測した値と実際の値の差（残差）を異常スコアとする。
スコアが正常期間の上位 5% を超えた点を異常と判定する。」

（模範解答 2）実務への応用：
- 工場：「機器温度の異常上昇を検知して予防保全アラートを出す」
- 売上：「前日比 +30% 以上の売上異常（過剰注文 or キャンペーン効果）を自動検知する」
- 在庫：「在庫数が急に減った日（盗難・集中注文）を検知する」

よくある受講生の詰まり方：
- 「しきい値はどう決めるの？」→「ビジネス的に許容できる誤検知率で決める。今回は上位 5%（=誤検知 5%）」

---
## 問4（発展）  しきい値を変えて比較する ★★★

📋 発展 時間が余った人だけ取り組んでください

---

Q：パーセンタイルを 90・95・99 で変えて、異常判定件数がどう変わるか比較してください

💡 ヒント Copilot への相談例：
「for ループを使って複数のパーセンタイル値でしきい値を変えて、それぞれの異常件数を出力したい」

In [ ]:
# パーセンタイルを変えながら異常件数の変化を比較する
for pct in [90, 95, 99]:
    thr = np.percentile(normal_score, pct)
    cnt = (df["score"] > thr).sum()
    print(f"パーセンタイル {pct}%: しきい値={thr:.2f}  異常件数={cnt} 件")

---
### 📊 出力結果の解説

```
パーセンタイル 90%: しきい値=2.87  異常件数=17 件
パーセンタイル 95%: しきい値=4.32  異常件数= 9 件
パーセンタイル 99%: しきい値=7.14  異常件数= 5 件
```

| パーセンタイル | 効果 | トレードオフ |
|-------------|------|-----------|
| 90%（低い） | より多くを異常と判定（見逃し少） | 誤検知が増える |
| 99%（高い） | 明確な異常だけを検出（誤検知少） | 見逃しが増える |

📌 ポイント 「見逃し（Recall 低下）」と「誤検知（Precision 低下）」のトレードオフ。
医療診断（見逃しが致命的）と不正検知（誤検知が多いとコスト増）では最適なしきい値が違う。

✅ （模範解答）90% では 17件の異常（正常期間内のノイズも含む）、99% では 5 件（明確なスパイクのみ）。
ビジネス要件に応じてしきい値を調整する。

---
## お疲れさまでした！

| 操作 | 発見 | ビジネスへの接続 |
|------|------|----------------|
| 可視化 | time 80〜90 にスパイク | 目視でも確認できた |
| ラグ特徴量 | 過去の値を入力として使う | 時系列への機械学習の基本 |
| 残差計算 | 正常モデルとのズレを数値化 | 異常スコアの考え方 |
| しきい値判定 | 9 件の異常を自動検出 | 工場監視・売上モニタリングで応用可 |